# Deploy `google/gemma-4-31B-it` with MLflow PyFunc to Model Serving

This notebook registers the **Gemma 4 31B Instruct** model from HuggingFace to **Unity Catalog** (`catalog.schema`) using MLflow `pyfunc`, then deploys it to a GPU-backed Model Serving endpoint via the Databricks SDK.

## Install and import libraries 

In [0]:
%pip install --upgrade mlflow "transformers>=4.45,<5.0" accelerate "torch>=2.0,<2.12"
dbutils.library.restartPython()

In [0]:
import os
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_registry_uri("databricks-uc")

## Initialize and configure your model

Download your model and package it with the model container.

In [0]:
import torch

model_id = "google/gemma-4-31B-it"

print(f"Target model: {model_id}")
print(f"Torch version: {torch.__version__}, CUDA available: {torch.cuda.is_available()}")

In [0]:
class GemmaModel(mlflow.pyfunc.PythonModel):
    """Custom PyFunc wrapper for google/gemma-4-31B-it.
    Downloads and loads the model at serving time (on GPU endpoint).
    """

    def load_context(self, context):
        import torch
        from transformers import pipeline as hf_pipeline

        with open(context.artifacts["model_config"], "r") as f:
            self.model_id = f.read().strip()

        self.generator = hf_pipeline(
            "text-generation",
            model=self.model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto",
        )

    def predict(self, context, model_input, params=None):
        """Generate text from prompts using chat template."""
        max_new_tokens = (params or {}).get("max_new_tokens", 256)
        temperature = (params or {}).get("temperature", 0.7)

        prompts = model_input["prompt"].tolist()
        outputs = []
        for prompt in prompts:
            messages = [{"role": "user", "content": prompt}]
            result = self.generator(
                messages,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=True,
            )
            generated = result[0]["generated_text"]
            if isinstance(generated, list):
                assistant_msg = generated[-1]["content"]
            else:
                assistant_msg = generated[len(prompt):].strip()
            outputs.append(assistant_msg)
        return outputs

## Log your model using MLflow

The following code defines the inference parameters to pass to the model at the time of inference and the model schema.

In [0]:
from mlflow.models import infer_signature, ModelSignature
from mlflow.types.schema import Schema, ColSpec, ParamSchema, ParamSpec

input_schema = Schema([ColSpec("string", "prompt")])
output_schema = Schema([ColSpec("string", "generated_text")])
param_schema = ParamSchema([
    ParamSpec("max_new_tokens", "long", 256),
    ParamSpec("temperature", "double", 0.7),
])

signature = ModelSignature(
    inputs=input_schema,
    outputs=output_schema,
    params=param_schema,
)

print("Model signature defined:")
print(signature)

Log the model with the MLflow `pyfunc` flavor.

In [0]:
import tempfile

registered_model_name = "malay_demo.sample.gemma_4_31b_it" # Update corresponding registered model name

model_config_path = os.path.join(tempfile.mkdtemp(), "model_id.txt")
with open(model_config_path, "w") as f:
    f.write(model_id)

with mlflow.start_run():
    model_info = mlflow.pyfunc.log_model(
        name="model",
        python_model=GemmaModel(),
        artifacts={"model_config": model_config_path},
        pip_requirements=["torch", "transformers>=5.0", "accelerate"],
        input_example=pd.DataFrame({"prompt": ["Explain quantum computing in simple terms."]}),
        signature=signature,
        registered_model_name=registered_model_name,
    )
    print(f"Model logged: {model_info.model_uri}")
    print(f"Registered as: {registered_model_name}")

## Test your model in a notebook

In [0]:
client = MlflowClient()
versions = client.search_model_versions(f"name='{registered_model_name}'")
latest = versions[0]

print(f"Registered model: {latest.name}")
print(f"Version: {latest.version}")
print(f"Status: {latest.status}")
print(f"Source: {latest.source}")

## Configure and create your model serving endpoint

The following variables set the values for configuring the model serving endpoint, such as the endpoint name, compute type, and which model to serve with the endpoint. After you call the create endpoint API, the logged model is deployed to the endpoint.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    ServedModelInputWorkloadType,
)

w = WorkspaceClient()

endpoint_name = "gemma-4-31b-it"
model_name = "malay_demo.sample.gemma_4_31b_it" # Update corresponding registered model name
model_version = latest.version

# GPU_LARGE required for 31B parameter model
workload_type = ServedModelInputWorkloadType.GPU_LARGE
workload_size = "Small"
scale_to_zero = False

print(f"Endpoint: {endpoint_name}")
print(f"Model: {model_name} v{model_version}")
print(f"Workload: {workload_type}, Size: {workload_size}")

The following sends the POST request to create the serving endpoint.

In [0]:
w.serving_endpoints.create(
    name=endpoint_name,
    config=EndpointCoreConfigInput(
        served_entities=[
            ServedEntityInput(
                entity_name=model_name,
                entity_version=model_version,
                workload_size=workload_size,
                scale_to_zero_enabled=scale_to_zero,
                workload_type=workload_type,
            )
        ]
    ),
)

print(f"Endpoint '{endpoint_name}' creation initiated. Model: {model_name} v{model_version}")

## View your endpoint
To see more information about your endpoint, go to the **Serving** UI and search for your endpoint name.

## Query your endpoint

Once your endpoint is ready, you can query it by making an API request. Depending on the model size and complexity, it can take 10-15 minutes or more for the endpoint to get ready.  

In [0]:
response = w.serving_endpoints.query(
    name=endpoint_name,
    dataframe_records=[{
        "prompt": "Tell me some of basic concepts of quantum computing.",
    }],
)

print(response.predictions)